# 🎁 GiftGenius — Kaggle backend на Qwen3.5-4B (4-bit)

Этот ноутбук запускает **весь LLM-инференс без внешнего API** прямо в Kaggle на **одной GPU T4**.  
Локально у вас остается только Streamlit-интерфейс.

Схема:
- **локально**: `app.py` / `app_qwen_local.py`
- **в Kaggle**: `FastAPI + RAG + Qwen3.5-4B 4-bit`
- связь между ними: `ngrok`


In [1]:
pip cache purge

Files removed: 0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# !pip uninstall -y bitsandbytes transformers accelerate

In [1]:
# Установка всех необходимых библиотек
!pip install -q \
  transformers \
  accelerate \
  bitsandbytes==0.48.2 \
  langgraph \
  langchain-core \
  langgraph-checkpoint

# Установка pyngrok для туннелирования
!pip install -q pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 32.4 MB/s eta 0:00:00:00:0100:01


In [2]:
try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes version: {bnb.__version__}")
except Exception as e:
    print(f"❌ bitsandbytes error: {e}")

✅ bitsandbytes version: 0.48.2


In [3]:
# 3. Проверим, что NumPy загрузился правильно
import numpy as np
import pandas as pd

print(f"NumPy version: {np.__version__}")
print(f"NumPy modules: {[m for m in dir(np) if not m.startswith('_')][:10]}")
print(f"Pandas version: {pd.__version__}")

# Проверяем наличие numpy.rec
try:
    from numpy import rec
    print("✅ numpy.rec доступен")
except ImportError as e:
    print("❌ numpy.rec НЕ доступен")
    print(f"Ошибка: {e}")

NumPy version: 2.0.2
NumPy modules: ['False_', 'ScalarType', 'True_', 'abs', 'absolute', 'acos', 'acosh', 'add', 'all', 'allclose']
Pandas version: 2.3.3
✅ numpy.rec доступен


In [4]:
import numpy as np
import pandas as pd
import torch

print("📊 Стандартные библиотеки Kaggle:")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}") 
print(f"PyTorch version: {torch.__version__}")
print("✅ Окружение в норме")

📊 Стандартные библиотеки Kaggle:
NumPy version: 2.0.2
Pandas version: 2.3.3
PyTorch version: 2.9.0+cu126
✅ Окружение в норме


In [10]:
%%writefile rag_pipeline_qwen_kaggle.py
import math
import os
import re
import urllib.parse
import json
from collections import Counter
from typing import Any, Dict, List, Literal, Optional, Tuple, TypedDict, Annotated
import operator

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from langgraph.constants import END, START
    from langgraph.graph import StateGraph
    LANGGRAPH_AVAILABLE = True
except Exception:
    LANGGRAPH_AVAILABLE = False
    START = "__start__"
    END = "__end__"


# ============================================================
# Config
# ============================================================
NGROK_AUTH_TOKEN = "3B5OF8kQosYZeZOwLm6Sf56Mh9S_6mGQPsT52NNSa5x2CPKez"
MODEL_NAME = "Qwen/Qwen3.5-4B"
DATA_PATH = "/kaggle/input/datasets/ursofiia/gg-database/gifts_dataset_ru_plus.csv"
FINAL_TOP_K = 20
IDEAS_TO_GENERATE = 10
MAX_GENERATION_RETRIES = 1
ENABLE_THINKING = False
LLM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GEN_CONFIG = {
    "temperature": 0.65,
    "top_p": 0.9,
    "top_k": 50,
    "repetition_penalty": 1.08,
}


# ============================================================
# Helpers
# ============================================================
def normalize_text(text: str) -> str:
    text = str(text or "")
    text = text.replace("ё", "е")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_product_name(name: str) -> str:
    name = re.sub(r"[‑–—]", "-", normalize_text(name))
    return name


def create_market_links(product_name: str) -> dict:
    clean_name = normalize_product_name(product_name.split(",")[0].strip().strip('"'))
    encoded_wb = urllib.parse.quote(clean_name)
    encoded_ya = urllib.parse.quote(clean_name).replace("%20", "+")
    encoded_ozon = urllib.parse.quote(clean_name)
    return {
        "wildberries": f"https://www.wildberries.ru/catalog/0/search.aspx?search={encoded_wb}",
        "yamarket": f"https://market.yandex.ru/search?text={encoded_ya}",
        "ozon": f"https://www.ozon.ru/search/?text={encoded_ozon}",
    }


# ============================================================
# Dataset + BM25 retrieval
# ============================================================
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"DATA_PATH не найден: {DATA_PATH}")

df = pd.read_csv(DATA_PATH).fillna("")
print(f"✅ Загружено {len(df)} подарков")
df["_doc"] = df.astype(str).agg(" | ".join, axis=1).str.strip()
corpus = df["_doc"].tolist()


def tokenize(text: str) -> List[str]:
    text = normalize_text(text).lower()
    return re.findall(r"[a-zа-я0-9]+", text)


print("🔍 Построение BM25 индекса...")
corpus_tokens = [tokenize(doc) for doc in corpus]
doc_lens = [len(tokens) for tokens in corpus_tokens]
avgdl = sum(doc_lens) / max(len(doc_lens), 1)
doc_freq = Counter()
doc_term_freqs = []
for tokens in corpus_tokens:
    tf = Counter(tokens)
    doc_term_freqs.append(tf)
    for tok in tf.keys():
        doc_freq[tok] += 1

N = len(corpus_tokens)
IDF = {tok: math.log(1 + (N - dfreq + 0.5) / (dfreq + 0.5)) for tok, dfreq in doc_freq.items()}
K1 = 1.5
B = 0.75
print(f"✅ BM25 готов | docs={N} | avgdl={avgdl:.1f}")


def bm25_score(query_tokens: List[str], doc_idx: int) -> float:
    tf = doc_term_freqs[doc_idx]
    dl = doc_lens[doc_idx]
    score = 0.0
    for tok in query_tokens:
        freq = tf.get(tok, 0)
        if not freq:
            continue
        idf = IDF.get(tok, 0.0)
        denom = freq + K1 * (1 - B + B * dl / max(avgdl, 1e-9))
        score += idf * (freq * (K1 + 1)) / max(denom, 1e-9)
    return score


def retrieve_for_query(query_text: str, top_k: int = FINAL_TOP_K) -> List[int]:
    query_tokens = tokenize(query_text)
    if not query_tokens:
        return list(range(min(top_k, len(df))))

    raw_query = normalize_text(query_text).lower()
    scores = []
    for i, doc_text in enumerate(corpus):
        score = bm25_score(query_tokens, i)
        doc_lower = normalize_text(doc_text).lower()
        for tok in set(query_tokens):
            if tok in doc_lower:
                score += 0.12
        if raw_query and raw_query in doc_lower:
            score += 1.0
        scores.append((score, i))

    scores.sort(key=lambda x: x[0], reverse=True)
    return [i for _, i in scores[: max(top_k * 2, 20)]]


def get_products_context(idx_list: List[int], top_k: int = FINAL_TOP_K) -> Tuple[List[str], pd.DataFrame]:
    context_df = df.iloc[idx_list[:top_k]].copy()
    products = []
    for _, row in context_df.iterrows():
        name = str(row.iloc[0]).split(",")[0].strip().strip('"')
        products.append(name)
    return products, context_df


# ============================================================
# LLM
# ============================================================
print("🤖 Загрузка Qwen2.5-3B-Instruct в 4-bit...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
print("✅ Модель загружена")


def _apply_chat_template(messages: list[dict]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def call_local_llm(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 512,
    temperature: Optional[float] = None,
    top_p: Optional[float] = None,
    top_k: Optional[int] = None,
) -> str:
    temperature = GEN_CONFIG["temperature"] if temperature is None else temperature
    top_p = GEN_CONFIG["top_p"] if top_p is None else top_p
    top_k = GEN_CONFIG["top_k"] if top_k is None else top_k

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = _apply_chat_template(messages)
    inputs = tokenizer(prompt, return_tensors="pt")
    target_device = "cuda:0" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(target_device) for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=GEN_CONFIG["repetition_penalty"],
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
    text = tokenizer.batch_decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return text


# ============================================================
# Prompts
# ============================================================

VALIDATION_SYSTEM = """Ты — строгий валидатор запросов для сервиса подбора подарков.

Твоя задача — проверить, является ли запрос пользователя подходящим для подбора подарка.

ПРАВИЛА:
1. Запрос должен быть СВЯЗАН С ПОДАРКОМ (описывать человека, интересы, повод)
2. Если запрос — бессвязный набор букв (типа "аааа", "фыва", "12345") — ОТКЛОНИТЬ
3. Если запрос не по теме (рецепты, программирование, погода) — ОТКЛОНИТЬ
4. Во всех остальных случаях — ОДОБРИТЬ

Ответь строго в формате JSON:
{"is_valid": true/false, "reason": "причина отказа (только если false)"}
"""

EXTRACTION_SYSTEM = """Ты — аналитик запросов для сервиса подбора подарков.

Извлеки из запроса следующую информацию. ВАЖНО: извлекай ТОЛЬКО то, что явно указано в запросе! НЕ ДОДУМЫВАЙ!

1. recipient (получатель) — кому подарок (маме, папе, другу, коллеге, начальнику, соседу и т.д.)
2. occasion (повод) — по какому случаю (день рождения, новый год, просто так, повышение и т.д.)
3. interests (интересы) — список интересов/хобби получателя
4. age (возраст) — если указан
5. gender (пол) — если можно определить (male/female)
6. dislikes (что не любит) — список того, что человек НЕ ЛЮБИТ, НЕ ХОЧЕТ получать в подарок

Правила:
- Если информация отсутствует в запросе, ставь null или пустой массив
- Не додумывай возраст, пол или интересы, если они не указаны
- Извлекай ТОЛЬКО явно указанное
- Для interests и dislikes возвращай массив строк

Ответь строго в формате JSON:
{
  "recipient": "строка или null",
  "occasion": "строка или null",
  "interests": ["интерес1", "интерес2"],
  "age": число или null,
  "gender": "male/female/null",
  "dislikes": ["что не любит1", "что не любит2"],
  "data_extracted": true/false
}
"""

PERSONA_SYSTEM = """Ты — психолог и gift-strategist.
Составь краткий, но полезный портрет получателя подарка на основе предоставленной информации.

Нужно описать:
1. возраст/роль/тип получателя
2. образ жизни и интересы
3. что может его порадовать
4. какие ограничения есть в выборе подарка (что точно не нужно дарить)

Пиши по-русски, 3-5 предложений, без воды.
"""

CREATIVE_SYSTEM = f"""Ты — эксперт по подбору уникальных и персонализированных подарков.

Твоя задача — сгенерировать {IDEAS_TO_GENERATE} креативных идей подарков, которые идеально подходят конкретному человеку.

ВАЖНЫЕ ПРАВИЛА:
1. **Персонализация**: каждая идея должна быть привязана к личности получателя, его интересам и образу жизни
2. **Комбинация интересов**: если у человека несколько интересов, придумывай подарки, которые объединяют их
3. **Учет ограничений**: строго исключи подарки из списка "НЕ ЛЮБИТ"
4. **Оригинальность**: избегай банальностей (носки, кружки, стандартные наборы)
5. **Конкретность**: названия должны быть четкими и понятными для поиска на маркетплейсах

Формат для каждой идеи (строго соблюдай):
ПОДАРОК: [конкретное название]
ПОЧЕМУ: [почему это идеально подходит этому человеку]
МОМЕНТ: [в какой ситуации будет уместен]
ЧЕМ НЕ БАНАЛЬНО: [почему это не стандартный подарок]

Начинай сразу с первой идеи, без вступлений.
"""

FEEDBACK_ANALYSIS_SYSTEM = """Ты — аналитик обратной связи по подаркам.

Пользователь дал обратную связь на предыдущие идеи:
"{feedback}"

Проанализируй обратную связь и выдели:
1. Что нужно ИСКЛЮЧИТЬ (категории, которые не подходят)
2. Что нужно ДОБАВИТЬ (новые интересы, предпочтения)

Верни строго в формате JSON:
{{
  "remove": ["категория1", "категория2"],
  "add": ["интерес1", "интерес2"]
}}
"""


# ============================================================
# Parsing outputs
# ============================================================
def parse_json_from_llm(response: str) -> Optional[Dict]:
    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception as e:
        print(f"⚠️ Ошибка парсинга JSON: {e}")
    return None


def persona_prompt(metadata: Dict) -> str:
    parts = []
    if metadata.get("recipient"):
        parts.append(f"Получатель: {metadata['recipient']}")
    if metadata.get("age"):
        parts.append(f"Возраст: {metadata['age']}")
    if metadata.get("gender"):
        parts.append(f"Пол: {metadata['gender']}")
    if metadata.get("occasion"):
        parts.append(f"Повод: {metadata['occasion']}")
    if metadata.get("interests"):
        parts.append(f"Интересы: {', '.join(metadata['interests'])}")
    if metadata.get("dislikes"):
        parts.append(f"НЕ ЛЮБИТ: {', '.join(metadata['dislikes'])}")
    return "Информация о получателе:\n" + "\n".join(parts) if parts else "Информация о получателе не указана"


def creative_prompt(
    query: str,
    persona: str,
    metadata: Dict,
    products: List[str],
    feedback_context: Optional[str] = None,
) -> str:
    recipient_info = []
    if metadata.get("recipient"):
        recipient_info.append(f"Получатель: {metadata['recipient']}")
    if metadata.get("age"):
        recipient_info.append(f"Возраст: {metadata['age']}")
    if metadata.get("occasion"):
        recipient_info.append(f"Повод: {metadata['occasion']}")
    
    interests_text = ""
    if metadata.get("interests"):
        interests_text = f"\nИНТЕРЕСЫ: {', '.join(metadata['interests'])}"
    
    dislikes_text = ""
    if metadata.get("dislikes"):
        dislikes_text = f"\nНЕЖЕЛАТЕЛЬНЫЕ ПОДАРКИ (исключить): {', '.join(metadata['dislikes'])}"
    
    feedback_text = ""
    if feedback_context:
        feedback_text = f"\nУЧТИ ОБРАТНУЮ СВЯЗЬ ПОЛЬЗОВАТЕЛЯ:\n{feedback_context}\n"
    
    products_list = "\n".join([f"- {p}" for p in products[:15]])
    
    return f"""ПОРТРЕТ ПОЛУЧАТЕЛЯ:
{persona}

{chr(10).join(recipient_info) if recipient_info else ''}{interests_text}{dislikes_text}{feedback_text}

⚠️ ВАЖНО: Список ниже — это ТОЛЬКО ПРИМЕРЫ КАТЕГОРИЙ для вдохновения.
НЕ КОПИРУЙ ЭТИ ТОВАРЫ ДОСЛОВНО! Используй их чтобы понять, какие категории существуют, и придумай СВОИ уникальные идеи на стыке интересов человека.

КАТЕГОРИИ ПОДАРКОВ (для вдохновения):
{products_list}

ЗАДАЧА:
Создай {IDEAS_TO_GENERATE} персонализированных идей подарков для этого человека.

Начинай сразу с первой идеи в формате:
ПОДАРОК: ...
ПОЧЕМУ: ...
МОМЕНТ: ...
ЧЕМ НЕ БАНАЛЬНО: ..."""


def parse_ideas(creative_response: str) -> List[str]:
    ideas: List[str] = []
    current_idea: List[str] = []
    in_idea = False
    
    for raw_line in creative_response.split("\n"):
        line = raw_line.strip()
        if not line:
            continue
        
        is_new_idea = False
        
        if line.startswith("ПОДАРОК:"):
            is_new_idea = True
        elif re.match(r'^\d+\.', line):
            is_new_idea = True
            line = re.sub(r'^\d+\.\s*', '', line)
        elif line.startswith("ПОЧЕМУ:") and not in_idea:
            is_new_idea = True
        
        if is_new_idea:
            if current_idea:
                ideas.append("\n".join(current_idea))
                current_idea = []
            in_idea = True
            current_idea.append(line)
        elif line.startswith(("ПОЧЕМУ:", "МОМЕНТ:", "ЧЕМ НЕ БАНАЛЬНО:")):
            if in_idea:
                current_idea.append(line)
            else:
                in_idea = True
                current_idea.append(line)
        elif current_idea:
            current_idea.append(line)
    
    if current_idea:
        ideas.append("\n".join(current_idea))
    
    return ideas


def extract_idea_title(idea_text: str) -> str:
    for line in idea_text.splitlines():
        if line.startswith("ПОДАРОК:"):
            title = line.replace("ПОДАРОК:", "").strip()
            return title.strip('*').strip('"')
        match = re.match(r'^\d+\.\s*(.+)$', line)
        if match:
            return match.group(1).strip()
    
    for line in idea_text.splitlines():
        line = line.strip()
        if line and not line.startswith(("ПОЧЕМУ:", "МОМЕНТ:", "ЧЕМ НЕ БАНАЛЬНО:")):
            return line[:100] if len(line) > 100 else line
    return ""


# ============================================================
# Graph state
# ============================================================
class GiftGraphState(TypedDict, total=False):
    raw_query: str
    clarification_question: str
    recipient: Optional[str]
    occasion: Optional[str]
    age: Optional[int]
    gender: Optional[str]
    persona: str
    status: str
    user_feedback: Optional[str]
    feedback_context: str
    
    query_valid: bool
    metadata_extracted: bool
    
    interests: Annotated[List[str], operator.add]
    dislikes: Annotated[List[str], operator.add]
    retrieved_products: Annotated[List[str], operator.add]
    references: Annotated[List[str], operator.add]
    draft_ideas: Annotated[List[str], operator.add]
    final_ideas: Annotated[List[str], operator.add]
    debug_trace: Annotated[List[str], operator.add]
    
    search_links: Dict[str, Dict[str, str]]
    
    iteration: int
    max_iterations: int


# ============================================================
# Nodes (Агенты)
# ============================================================

def validate_input_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 1: Базовая валидация - только проверка длины"""
    query = normalize_text(state.get("raw_query", ""))
    
    if not query or len(query) < 3:
        return {
            "query_valid": False,
            "status": "needs_clarification",
            "clarification_question": "Пожалуйста, опишите запрос подробнее (минимум 3 символа).",
            "debug_trace": ["validate_input: слишком короткий запрос"]
        }
    
    return {
        "query_valid": True,
        "status": "ok",
        "debug_trace": ["validate_input: ok"]
    }


def extract_metadata_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 2: Извлечение всех данных из запроса"""
    query = state["raw_query"]
    
    extraction_response = call_local_llm(
        EXTRACTION_SYSTEM,
        f"Извлеки информацию из запроса: \"{query}\"",
        max_new_tokens=250,
        temperature=0.1
    )
    
    metadata = parse_json_from_llm(extraction_response)
    
    if metadata:
        print(f"  📋 Извлечено: получатель={metadata.get('recipient')}, повод={metadata.get('occasion')}, "
              f"интересы={metadata.get('interests', [])}, не любит={metadata.get('dislikes', [])}")
        
        return {
            "metadata_extracted": metadata.get("data_extracted", False),
            "recipient": metadata.get("recipient"),
            "occasion": metadata.get("occasion"),
            "interests": metadata.get("interests", []),
            "age": metadata.get("age"),
            "gender": metadata.get("gender"),
            "dislikes": metadata.get("dislikes", []),
            "debug_trace": ["extract_metadata: done"]
        }
    
    return {
        "metadata_extracted": False,
        "recipient": None,
        "occasion": None,
        "interests": [],
        "age": None,
        "gender": None,
        "dislikes": [],
        "debug_trace": ["extract_metadata: not extracted"]
    }


def llm_validation_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 3: LLM-валидация (только если данные не извлечены)"""
    query = state["raw_query"]
    
    validation_response = call_local_llm(
        VALIDATION_SYSTEM,
        f"Проверь этот запрос: \"{query}\"",
        max_new_tokens=100,
        temperature=0.1
    )
    
    result = parse_json_from_llm(validation_response)
    
    if result and not result.get("is_valid", True):
        reason = result.get("reason", "некорректный запрос")
        return {
            "query_valid": False,
            "status": "needs_clarification",
            "clarification_question": f"Запрос не подходит для подбора подарка: {reason}. Пожалуйста, опишите человека и его интересы.",
            "debug_trace": [f"llm_validation: отклонён - {reason}"]
        }
    
    return {
        "query_valid": True,
        "debug_trace": ["llm_validation: одобрен"]
    }


def create_persona_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 4: Создание портрета на основе извлеченных данных"""
    metadata = {
        "recipient": state.get("recipient"),
        "occasion": state.get("occasion"),
        "interests": state.get("interests", []),
        "age": state.get("age"),
        "gender": state.get("gender"),
        "dislikes": state.get("dislikes", [])
    }
    
    persona = call_local_llm(
        PERSONA_SYSTEM,
        persona_prompt(metadata),
        max_new_tokens=300,
        temperature=0.6,
    )
    
    if not persona:
        persona = "Получатель любит персональные и продуманные подарки, связанные с его образом жизни и интересами."
    
    return {
        "persona": persona,
        "debug_trace": ["create_persona: created"]
    }


def retrieve_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 5: Поиск товаров в базе (RAG)"""
    search_parts = [state["raw_query"]]
    
    if state.get("occasion"):
        search_parts.append(state["occasion"])
    if state.get("interests"):
        search_parts.extend(state["interests"])
    
    query = " ".join(search_parts).strip()
    
    idx = retrieve_for_query(query)
    products, _ = get_products_context(idx)
    
    return {
        "retrieved_products": products[:FINAL_TOP_K],
        "references": products[:15],
        "debug_trace": [f"retrieve: найдено {len(products[:FINAL_TOP_K])} референсов"]
    }


def generate_gifts_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 6: Генерация идей с учетом всех данных"""
    metadata = {
        "recipient": state.get("recipient"),
        "occasion": state.get("occasion"),
        "interests": state.get("interests", []),
        "age": state.get("age"),
        "gender": state.get("gender"),
        "dislikes": state.get("dislikes", [])
    }
    
    feedback_context = state.get("feedback_context", "")
    
    creative_response = call_local_llm(
        CREATIVE_SYSTEM,
        creative_prompt(
            query=state["raw_query"],
            persona=state["persona"],
            metadata=metadata,
            products=state.get("retrieved_products", []),
            feedback_context=feedback_context,
        ),
        max_new_tokens=1500,
        temperature=0.65,
    )
    
    ideas = parse_ideas(creative_response)
    
    filtered_ideas = []
    dislikes = state.get("dislikes", [])
    
    for idea in ideas:
        idea_lower = idea.lower()
        should_skip = False
        for dislike in dislikes:
            if dislike and dislike.lower() in idea_lower:
                should_skip = True
                print(f"  ⚠️ Исключена идея (нарушает dislike '{dislike}'): {extract_idea_title(idea)[:50]}")
                break
        if not should_skip:
            filtered_ideas.append(idea)
    
    if len(filtered_ideas) < 3:
        filtered_ideas = ideas[:IDEAS_TO_GENERATE]
    
    return {
        "draft_ideas": filtered_ideas[:IDEAS_TO_GENERATE],
        "debug_trace": [f"generate_gifts: {len(filtered_ideas[:IDEAS_TO_GENERATE])} идей"]
    }


def process_feedback_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 7: Обработка обратной связи и обновление контекста"""
    user_feedback = state.get("user_feedback", "")
    
    if not user_feedback:
        return {"debug_trace": ["process_feedback: нет обратной связи"]}
    
    print(f"  💬 Обработка обратной связи: {user_feedback}")
    
    feedback_prompt = FEEDBACK_ANALYSIS_SYSTEM.format(feedback=user_feedback)
    
    feedback_response = call_local_llm(
        "Ты — аналитик обратной связи по подаркам.",
        feedback_prompt,
        max_new_tokens=200,
        temperature=0.3
    )
    
    feedback_data = parse_json_from_llm(feedback_response)
    
    result = {}
    
    if feedback_data:
        remove_items = feedback_data.get("remove", [])
        if remove_items:
            result["dislikes"] = remove_items
            print(f"    ➖ Добавлено в исключения: {remove_items}")
        
        add_items = feedback_data.get("add", [])
        if add_items:
            result["interests"] = add_items
            print(f"    ➕ Добавлено в интересы: {add_items}")
    
    feedback_context = state.get("feedback_context", "")
    result["feedback_context"] = f"{feedback_context}\n[Обратная связь {state.get('iteration', 0) + 1}]: {user_feedback}"
    result["iteration"] = state.get("iteration", 0) + 1
    result["debug_trace"] = [f"process_feedback: обработана итерация {result['iteration']}"]
    
    return result


def generate_links_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 8: Генерация ссылок"""
    links: Dict[str, Dict[str, str]] = {}
    
    for idea in state.get("draft_ideas", [])[:IDEAS_TO_GENERATE]:
        title = extract_idea_title(idea)
        if title:
            links[title] = create_market_links(title)
    
    return {
        "search_links": links,
        "debug_trace": [f"generate_links: {len(links)} ссылок"]
    }


def finalize_node(state: GiftGraphState) -> Dict[str, Any]:
    """Агент 9: Финализация"""
    return {
        "final_ideas": state.get("draft_ideas", [])[:IDEAS_TO_GENERATE],
        "status": "complete",
        "debug_trace": [f"finalize: {len(state.get('draft_ideas', [])[:IDEAS_TO_GENERATE])} идей"]
    }


# ============================================================
# Routing
# ============================================================
def route_after_validation(state: GiftGraphState) -> Literal["extract_metadata", END]:
    return "extract_metadata" if state.get("query_valid", False) else END


def route_after_extraction(state: GiftGraphState) -> Literal["llm_validation", "create_persona"]:
    """Если данные извлечены -> создаем портрет, иначе -> валидация"""
    if state.get("metadata_extracted", False):
        return "create_persona"
    return "llm_validation"


def route_after_generation(state: GiftGraphState) -> Literal["process_feedback", "generate_links"]:
    if state.get("user_feedback") and state.get("iteration", 0) < state.get("max_iterations", 3):
        return "process_feedback"
    return "generate_links"


# ============================================================
# Graph builder - ИСПРАВЛЕННЫЙ (без параллельных веток!)
# ============================================================
def build_gift_graph():
    if not LANGGRAPH_AVAILABLE:
        return None
    
    graph = StateGraph(GiftGraphState)
    
    graph.add_node("validate_input", validate_input_node)
    graph.add_node("extract_metadata", extract_metadata_node)
    graph.add_node("llm_validation", llm_validation_node)
    graph.add_node("create_persona", create_persona_node)
    graph.add_node("retrieve", retrieve_node)
    graph.add_node("generate_gifts", generate_gifts_node)
    graph.add_node("process_feedback", process_feedback_node)
    graph.add_node("generate_links", generate_links_node)
    graph.add_node("finalize", finalize_node)

    # Начало
    graph.add_edge(START, "validate_input")
    graph.add_conditional_edges("validate_input", route_after_validation)
    
    # ✅ ИСПРАВЛЕНО: Только условное ребро, без параллельного edge
    graph.add_conditional_edges("extract_metadata", route_after_extraction)
    
    # ✅ Последовательная цепочка: llm_validation → create_persona → retrieve
    graph.add_edge("llm_validation", "create_persona")
    graph.add_edge("create_persona", "retrieve")
    
    # Дальнейшая последовательная обработка
    graph.add_edge("retrieve", "generate_gifts")
    graph.add_conditional_edges("generate_gifts", route_after_generation)
    graph.add_edge("process_feedback", "generate_gifts")
    graph.add_edge("generate_links", "finalize")
    graph.add_edge("finalize", END)
    
    return graph.compile()


gift_graph = build_gift_graph()


# ============================================================
# Sequential fallback
# ============================================================
def run_sequential(state: GiftGraphState) -> GiftGraphState:
    state = validate_input_node(state)
    if not state.get("query_valid"):
        return state
    
    state.update(extract_metadata_node(state))
    
    if not state.get("metadata_extracted", False):
        state.update(llm_validation_node(state))
        if not state.get("query_valid"):
            return state
    
    state.update(create_persona_node(state))
    state.update(retrieve_node(state))
    state.update(generate_gifts_node(state))
    
    if state.get("user_feedback") and state.get("iteration", 0) < state.get("max_iterations", 3):
        state.update(process_feedback_node(state))
        state.update(generate_gifts_node(state))
    
    state.update(generate_links_node(state))
    state.update(finalize_node(state))
    
    return state


# ============================================================
# Public API
# ============================================================
def gift_agent(query: str, verbose: bool = False, user_feedback: Optional[str] = None, 
               iteration: int = 0) -> Dict[str, Any]:
    """Главный агент для подбора подарков с поддержкой диалога."""
    print(f"🔍 GIFT_AGENT START: '{query}' (итерация {iteration})")
    
    initial_state: GiftGraphState = {
        "raw_query": query,
        "interests": [],
        "dislikes": [],
        "feedback_context": "",
        "debug_trace": [],
        "status": "started",
        "user_feedback": user_feedback,
        "iteration": iteration,
        "max_iterations": 3,
        "metadata_extracted": False,
        "query_valid": True,
        "retrieved_products": [],
        "references": [],
        "draft_ideas": [],
        "final_ideas": [],
        "search_links": {},
    }
    
    if len(query.strip()) < 3:
        print("❌ Слишком короткий запрос (<3 символов)")
        return {
            "status": "needs_clarification",
            "questions": "Пожалуйста, опишите запрос подробнее (минимум 3 символа).",
            "debug_trace": [],
        }
    
    print("🚀 Запуск графа...")
    result = gift_graph.invoke(initial_state) if gift_graph is not None else run_sequential(initial_state)
    
    print(f"📊 Результат: status={result.get('status')}")

    if result.get("status") == "needs_clarification":
        return {
            "status": "needs_clarification",
            "questions": result.get("clarification_question", "Уточните запрос."),
            "debug_trace": result.get("debug_trace", []),
        }

    payload = {
        "status": result.get("status", "success"),
        "ideas": result.get("final_ideas", []),
        "recipient": result.get("recipient"),
        "occasion": result.get("occasion"),
        "interests": result.get("interests", []),
        "dislikes": result.get("dislikes", []),
        "age": result.get("age"),
        "gender": result.get("gender"),
        "persona": result.get("persona", ""),
        "references": result.get("references", []),
        "search_links": result.get("search_links", {}),
        "debug_trace": result.get("debug_trace", []),
        "graph_mode": "langgraph" if gift_graph is not None else "sequential_fallback",
        "iteration": result.get("iteration", iteration),
        "can_continue": result.get("iteration", iteration) < 3,
    }
    
    print(f"✅ Успешно сгенерировано {len(payload['ideas'])} идей")
    print(f"   Итерация: {payload['iteration']}/3")
    
    if verbose:
        return payload
    
    payload.pop("debug_trace", None)
    return payload


def generate_gifts(description: str) -> List[str]:
    """Простая генерация без обратной связи"""
    result = gift_agent(description, verbose=False)
    
    if result["status"] == "needs_clarification":
        return [f"Уточните: {result['questions']}"]
    
    return result.get("ideas", ["Идеи не найдены"])


def generate_gifts_with_feedback(description: str, feedback: Optional[str] = None, 
                                  iteration: int = 0) -> Dict[str, Any]:
    """Генерация подарков с поддержкой обратной связи"""
    return gift_agent(description, verbose=False, user_feedback=feedback, iteration=iteration)


print("✅ RAG pipeline с оптимизированными агентами готов")
print(f"✅ Модель: {MODEL_NAME}")
print(f"✅ Агенты: валидация длины → извлечение данных → (LLM-валидация при необходимости) → портрет → RAG → генерация → обратная связь")
print(f"✅ Поддержка диалога: до 3 итераций с накоплением контекста")

Overwriting rag_pipeline_qwen_kaggle.py


In [6]:
%%writefile api_qwen_kaggle.py
from fastapi import FastAPI
from pydantic import BaseModel
import logging
import time

# Импортируем необходимые функции
from rag_pipeline_qwen_kaggle import generate_gifts, gift_agent

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(
    title="GiftGenius API | Qwen on Kaggle | Agent Pipeline",
    version="1.0.0",
)


class Request(BaseModel):
    description: str


class FeedbackRequest(BaseModel):
    description: str
    feedback: str | None = None
    iteration: int = 0


class Response(BaseModel):
    ideas: list[str]
    processing_time: float | None = None
    status: str = "success"
    graph_mode: str | None = None


class FeedbackResponse(BaseModel):
    ideas: list[str]
    processing_time: float | None = None
    status: str = "success"
    graph_mode: str | None = None
    iteration: int = 0
    can_continue: bool = True


@app.get("/")
def root():
    return {
        "message": "🎁 GiftGenius API | Qwen on Kaggle | Agent Pipeline",
        "status": "healthy",
        "endpoints": ["/health", "/generate", "/debug", "/generate_with_feedback"],
    }


@app.get("/health")
def health():
    return {"status": "healthy", "timestamp": time.time()}


@app.post("/generate", response_model=Response)
def generate(request: Request):
    start = time.time()
    logger.info("📨 Запрос: %s", request.description)

    try:
        result = gift_agent(request.description, verbose=False)

        if result.get("status") == "needs_clarification":
            return {
                "ideas": [f"Уточните: {result.get('questions', 'Недостаточно данных для подбора подарка.')}"],
                "processing_time": time.time() - start,
                "status": "needs_clarification",
                "graph_mode": result.get("graph_mode"),
            }

        return {
            "ideas": result.get("ideas", []),
            "processing_time": time.time() - start,
            "status": result.get("status", "success"),
            "graph_mode": result.get("graph_mode"),
        }
    except Exception as e:
        logger.exception("❌ Ошибка generate")
        return {
            "ideas": [f"Ошибка: {e}"],
            "processing_time": time.time() - start,
            "status": "error",
            "graph_mode": None,
        }


@app.post("/generate_with_feedback", response_model=FeedbackResponse)
def generate_with_feedback(request: FeedbackRequest):
    """
    Генерация подарков с возможностью обратной связи
    
    Args:
        request.description: Запрос пользователя
        request.feedback: Обратная связь от пользователя (что добавить/убрать)
        request.iteration: Номер итерации диалога
    """
    start = time.time()
    logger.info(f"📨 Запрос: {request.description}, обратная связь: {request.feedback}, итерация: {request.iteration}")
    
    try:
        result = gift_agent(
            request.description, 
            verbose=False,
            user_feedback=request.feedback,
            iteration=request.iteration
        )
        
        return {
            "ideas": result.get("ideas", []),
            "processing_time": time.time() - start,
            "status": result.get("status", "success"),
            "graph_mode": result.get("graph_mode"),
            "iteration": result.get("iteration", request.iteration),
            "can_continue": result.get("can_continue", True),
        }
    except Exception as e:
        logger.exception("❌ Ошибка generate_with_feedback")
        return {
            "ideas": [f"Ошибка: {e}"],
            "processing_time": time.time() - start,
            "status": "error",
            "graph_mode": None,
            "iteration": request.iteration,
            "can_continue": False,
        }


@app.post("/debug")
def debug(request: Request):
    start = time.time()
    logger.info("🛠 DEBUG запрос: %s", request.description)

    try:
        result = gift_agent(request.description, verbose=True)
        result["processing_time"] = time.time() - start
        return result
    except Exception as e:
        logger.exception("❌ Ошибка debug")
        return {
            "status": "error",
            "error": str(e),
            "processing_time": time.time() - start,
        }


# Для локального запуска внутри ноутбука / контейнера:
# !uvicorn api_qwen_kaggle:app --host 0.0.0.0 --port 8000

Writing api_qwen_kaggle.py


In [7]:
import torch
import traceback

print("GPU available:", torch.cuda.is_available())

try:
    import rag_pipeline_qwen_kaggle
    print("✅ rag_pipeline_qwen_kaggle импортирован успешно")
    
    # Проверяем, что токен и пути доступны
    print(f"📁 DATA_PATH: {rag_pipeline_qwen_kaggle.DATA_PATH}")
    print(f"🤖 MODEL_NAME: {rag_pipeline_qwen_kaggle.MODEL_NAME}")
    print(f"🔑 NGROK_TOKEN: {rag_pipeline_qwen_kaggle.NGROK_AUTH_TOKEN[:10]}...")
    
except Exception as e:
    print("❌ Ошибка при импорте rag_pipeline_qwen_kaggle")
    traceback.print_exc()
    raise

GPU available: True
✅ Загружено 305 подарков
🔍 Построение BM25 индекса...
✅ BM25 готов | docs=305 | avgdl=32.4
🤖 Загрузка Qwen2.5-3B-Instruct в 4-bit...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Модель загружена
✅ RAG pipeline с оптимизированными агентами готов
✅ Модель: Qwen/Qwen2.5-3B-Instruct
✅ Агенты: валидация длины → извлечение данных → (LLM-валидация при необходимости) → портрет → RAG → генерация → обратная связь
✅ Поддержка диалога: до 3 итераций с накоплением контекста
✅ rag_pipeline_qwen_kaggle импортирован успешно
📁 DATA_PATH: /kaggle/input/datasets/ursofiia/gg-database/gifts_dataset_ru_plus.csv
🤖 MODEL_NAME: Qwen/Qwen2.5-3B-Instruct
🔑 NGROK_TOKEN: 3B5OF8kQos...


In [8]:
import requests
import uvicorn
from threading import Thread
import time
import os
import traceback

def run_server():
    try:
        uvicorn.run("api_qwen_kaggle:app", host="0.0.0.0", port=8000, log_level="info")
    except Exception:
        print("❌ Uvicorn crashed:")
        traceback.print_exc()

os.system("pkill -f uvicorn || true")
os.system("pkill -f ngrok || true")
time.sleep(2)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

ready = False
for step in range(120):  # до ~10 минут ожидания
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        print("✅ FastAPI сервер поднят:", r.json())
        ready = True
        break
    except Exception:
        if step % 6 == 0:
            print(f"⏳ Ждем запуск сервера... {step*5} сек")
        time.sleep(5)

if not ready:
    raise RuntimeError(
        "Сервер не поднялся. Смотрите traceback выше. "
        "Если ошибка уже не про scipy/sklearn, пришлите новый traceback."
    )


⏳ Ждем запуск сервера... 0 сек


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:49124 - "GET /health HTTP/1.1" 200 OK
✅ FastAPI сервер поднят: {'status': 'healthy', 'timestamp': 1774044865.8263142}


In [9]:
from pyngrok import ngrok
import rag_pipeline_qwen_kaggle  # импортируем для доступа к токену

# Используем токен из файла
ngrok.set_auth_token(rag_pipeline_qwen_kaggle.NGROK_AUTH_TOKEN)
ngrok.kill()

public_url = ngrok.connect(8000)
print("=" * 60)
print("🎁 GIFT GENIUS + QWEN2.5-3B ГОТОВ")
print("=" * 60)
print("🌐 BASE URL:", public_url.public_url)
print("📌 HEALTH:", f"{public_url.public_url}/health")
print("📌 GENERATE:", f"{public_url.public_url}/generate")
print("=" * 60)

INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
INFO:pyngrok.ngrok:Opening tunnel named: http-8000-afe8e21e-30b3-43ed-976a-1b56358167de
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="FIPS 140 mode" enabled=false
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg="tunnel sessi

🎁 GIFT GENIUS + QWEN2.5-3B ГОТОВ
🌐 BASE URL: https://walker-unerasable-will.ngrok-free.dev
📌 HEALTH: https://walker-unerasable-will.ngrok-free.dev/health
📌 GENERATE: https://walker-unerasable-will.ngrok-free.dev/generate


INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:32+0000 lvl=info msg=end pg=/api/tunnels id=e5ba9cb78a302412 status=201 dur=293.769372ms
INFO:pyngrok.process.ngrok:t=2026-03-20T22:14:55+0000 lvl=info msg="join connections" obj=join id=52da1736fe8b l=127.0.0.1:8000 r=121.127.46.69:38188


INFO:     121.127.46.69:0 - "GET /health HTTP/1.1" 200 OK


INFO:pyngrok.process.ngrok:t=2026-03-20T22:15:03+0000 lvl=info msg="join connections" obj=join id=e1f0f8abf788 l=127.0.0.1:8000 r=121.127.46.69:57932
INFO:api_qwen_kaggle:📨 Запрос: 👨 Папа, 55 лет, рыбалка, дача, обратная связь: None, итерация: 0


🔍 GIFT_AGENT START: '👨 Папа, 55 лет, рыбалка, дача' (итерация 0)
🚀 Запуск графа...
  📋 Извлечено: получатель=папа, повод=None, интересы=['рыбалка', 'дача'], не любит=[]
📊 Результат: status=complete
✅ Успешно сгенерировано 10 идей
   Итерация: 0/3
INFO:     121.127.46.69:0 - "POST /generate_with_feedback HTTP/1.1" 200 OK


INFO:pyngrok.process.ngrok:t=2026-03-20T22:19:36+0000 lvl=info msg="join connections" obj=join id=a67c9ad7e169 l=127.0.0.1:8000 r=121.127.46.69:30654
INFO:api_qwen_kaggle:📨 Запрос: 👨 Папа, 55 лет, рыбалка, дача, обратная связь: убери все что связано с компостером и добавь турецкие сладости, итерация: 1


🔍 GIFT_AGENT START: '👨 Папа, 55 лет, рыбалка, дача' (итерация 1)
🚀 Запуск графа...
  📋 Извлечено: получатель=папа, повод=None, интересы=['рыбалка', 'дача'], не любит=[]
  💬 Обработка обратной связи: убери все что связано с компостером и добавь турецкие сладости
    ➖ Добавлено в исключения: ['компостер']
    ➕ Добавлено в интересы: ['турецкие сладости']
  💬 Обработка обратной связи: убери все что связано с компостером и добавь турецкие сладости
    ➖ Добавлено в исключения: ['компостер']
    ➕ Добавлено в интересы: ['турецкие сладости']
📊 Результат: status=complete
✅ Успешно сгенерировано 10 идей
   Итерация: 3/3


In [11]:
%%writefile test_api.py
import requests
import time

print("🔍 Тестирование GiftGenius API...")
print("=" * 60)

# Ждём немного, чтобы сервер точно запустился
time.sleep(2)

# Проверяем health
try:
    health = requests.get("http://127.0.0.1:8000/health", timeout=10)
    print("✅ Health check:", health.status_code)
    print("   ", health.json())
except Exception as e:
    print(f"❌ Ошибка подключения к серверу: {e}")
    print("   Убедитесь, что сервер запущен (uvicorn api_qwen_kaggle:app)")
    exit(1)

# Тестовый запрос
test_query = "Маме 50 лет, любит готовить и читать. Не любит сладкое и украшения."
print(f"\n📝 Тестовый запрос: {test_query}")
print("=" * 60)

try:
    response = requests.post(
        "http://127.0.0.1:8000/generate",
        json={"description": test_query},
        timeout=360,
    )
    
    print(f"📊 Status: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        print(f"⏱️ Время обработки: {result.get('processing_time', 0):.2f} сек")
        print(f"🎁 Найдено идей: {len(result.get('ideas', []))}")
        
        for i, idea in enumerate(result.get("ideas", []), 1):
            print("\n" + "=" * 40)
            print(f"ИДЕЯ {i}")
            print("=" * 40)
            print(idea)
    else:
        print(f"❌ Ошибка: {response.text}")
        
except Exception as e:
    print(f"❌ Ошибка при выполнении запроса: {e}")

Writing test_api.py


In [12]:
!python test_api.py

🔍 Тестирование GiftGenius API...
INFO:     127.0.0.1:37296 - "GET /health HTTP/1.1" 200 OK


INFO:api_qwen_kaggle:📨 Запрос: Маме 50 лет, любит готовить и читать. Не любит сладкое и украшения.


🔍 GIFT_AGENT START: 'Маме 50 лет, любит готовить и читать. Не любит сладкое и украшения.' (итерация 0)
🚀 Запуск графа...
✅ Health check: 200
    {'status': 'healthy', 'timestamp': 1774041853.7519486}

📝 Тестовый запрос: Маме 50 лет, любит готовить и читать. Не любит сладкое и украшения.
  📋 Извлечено: получатель=маме, повод=None, интересы=['готовить', 'читать'], не любит=['сладкое', 'украшения']
📊 Результат: status=complete
✅ Успешно сгенерировано 10 идей
   Итерация: 0/3
INFO:     127.0.0.1:37308 - "POST /generate HTTP/1.1" 200 OK
📊 Status: 200
⏱️ Время обработки: 152.92 сек
🎁 Найдено идей: 10

ИДЕЯ 1
ПОДАРОК: Книжный короб
ПОЧЕМУ: Это идеальный подарок для любителя чтения. Он позволяет хранить и организовать всю коллекцию книг мамы в стильном и удобном контейнере. Можно добавить личную автографку от вас или подписи о любви.
МОМЕНТ: Когда мама открывает свой новый книжный короб после покупки новых книг.
ЧЕМ НЕ БАНАЛЬНО: Это оригинальный способ сохранить и организовать ее коллекцию кни

In [17]:
%%writefile test_dialog_api.py
import requests
import time
import json

print("🔍 Тестирование GiftGenius API с диалогом")
print("=" * 60)

BASE_URL = "http://127.0.0.1:8000"

# Проверяем, что сервер работает
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5)
    print("✅ Сервер работает:", health.json())
except Exception as e:
    print(f"❌ Сервер не отвечает: {e}")
    exit(1)

print("\n" + "=" * 60)
print("📝 НАЧАЛО ДИАЛОГА")
print("=" * 60)

query = "Подруге 20 лет, любит готовить и читать. Любит птиц.  Не любит сладкое и украшения."

# ============================================================
# ИТЕРАЦИЯ 1: Первый запрос (без обратной связи)
# ============================================================
print("\n🎯 ИТЕРАЦИЯ 1: Первый запрос (без обратной связи)")
print("-" * 40)

response1 = requests.post(
    f"{BASE_URL}/generate_with_feedback",
    json={
        "description": query,
        "feedback": None,
        "iteration": 0
    },
    timeout=360
)

if response1.status_code == 200:
    result1 = response1.json()
    print(f"⏱️ Время обработки: {result1.get('processing_time', 0):.2f} сек")
    print(f"🎁 Найдено идей: {len(result1.get('ideas', []))}")
    print(f"📌 Статус: {result1.get('status')}")
    print(f"🔄 Итерация: {result1.get('iteration')}")
    print(f"💬 Можно продолжить: {result1.get('can_continue')}")
    
    print(f"\n📋 Первые 3 идеи:")
    for i, idea in enumerate(result1.get('ideas', [])[:3], 1):
        print(f"\n--- ИДЕЯ {i} ---")
        lines = idea.split('\n')
        for line in lines[:2]:
            print(line[:100])
    
    # ============================================================
    # ИТЕРАЦИЯ 2: С обратной связью (продолжение диалога)
    # ============================================================
    print("\n" + "=" * 60)
    print("🎯 ИТЕРАЦИЯ 2: Уточнение подбора (с обратной связью)")
    print("-" * 40)
    
    # Обратная связь от пользователя
    feedback = "Убери всё, что связано с техникой. Добавь больше идей для кулинарии и чтения."
    
    print(f"💬 Обратная связь: \"{feedback}\"")
    
    response2 = requests.post(
        f"{BASE_URL}/generate_with_feedback",
        json={
            "description": query,
            "feedback": feedback,
            "iteration": 1  # ← Важно: iteration = 1
        },
        timeout=600
    )
    
    if response2.status_code == 200:
        result2 = response2.json()
        print(f"⏱️ Время обработки: {result2.get('processing_time', 0):.2f} сек")
        print(f"🎁 Найдено идей: {len(result2.get('ideas', []))}")
        print(f"📌 Статус: {result2.get('status')}")
        print(f"🔄 Итерация: {result2.get('iteration')}")
        print(f"💬 Можно продолжить: {result2.get('can_continue')}")
        
        print(f"\n📋 Обновлённые идеи (первые 3):")
        for i, idea in enumerate(result2.get('ideas', [])[:3], 1):
            print(f"\n--- ИДЕЯ {i} ---")
            lines = idea.split('\n')
            for line in lines[:2]:
                print(line[:100])
        
        # ============================================================
        # ИТЕРАЦИЯ 3: Ещё одно уточнение (если можно продолжить)
        # ============================================================
        if result2.get('can_continue'):
            print("\n" + "=" * 60)
            print("🎯 ИТЕРАЦИЯ 3: Дополнительное уточнение")
            print("-" * 40)
            
            feedback2 = "Добавь что-то для отдыха и релаксации, например, для ванны или чаепития."
            
            print(f"💬 Обратная связь: \"{feedback2}\"")
            
            response3 = requests.post(
                f"{BASE_URL}/generate_with_feedback",
                json={
                    "description": query,
                    "feedback": feedback2,
                    "iteration": 2
                },
                timeout=360
            )
            
            if response3.status_code == 200:
                result3 = response3.json()
                print(f"⏱️ Время обработки: {result3.get('processing_time', 0):.2f} сек")
                print(f"🎁 Найдено идей: {len(result3.get('ideas', []))}")
                print(f"📌 Статус: {result3.get('status')}")
                print(f"🔄 Итерация: {result3.get('iteration')}")
                print(f"💬 Можно продолжить: {result3.get('can_continue')}")
                
                print(f"\n📋 Финальные идеи:")
                for i, idea in enumerate(result3.get('ideas', []), 1):
                    print(f"\n--- ИДЕЯ {i} ---")
                    lines = idea.split('\n')
                    for line in lines:
                        if line.startswith("ПОДАРОК:"):
                            print(line)
                            break
    else:
        print(f"❌ Ошибка: {response2.text}")
else:
    print(f"❌ Ошибка: {response1.text}")

print("\n" + "=" * 60)
print("✅ ДИАЛОГ ЗАВЕРШЁН")
print("=" * 60)

Overwriting test_dialog_api.py


In [18]:
!python test_dialog_api.py

INFO:     127.0.0.1:38404 - "GET /health HTTP/1.1" 200 OK
🔍 Тестирование GiftGenius API с диалогом


INFO:api_qwen_kaggle:📨 Запрос: Подруге 20 лет, любит готовить и читать. Любит птиц.  Не любит сладкое и украшения., обратная связь: None, итерация: 0


🔍 GIFT_AGENT START: 'Подруге 20 лет, любит готовить и читать. Любит птиц.  Не любит сладкое и украшения.' (итерация 0)
🚀 Запуск графа...
✅ Сервер работает: {'status': 'healthy', 'timestamp': 1774043035.1838815}

📝 НАЧАЛО ДИАЛОГА

🎯 ИТЕРАЦИЯ 1: Первый запрос (без обратной связи)
----------------------------------------
  📋 Извлечено: получатель=подруге, повод=None, интересы=['готовить', 'читать', 'любит птиц'], не любит=['сладкое', 'украшения']
📊 Результат: status=complete
✅ Успешно сгенерировано 10 идей
   Итерация: 0/3
INFO:     127.0.0.1:38414 - "POST /generate_with_feedback HTTP/1.1" 200 OK
⏱️ Время обработки: 167.52 сек
🎁 Найдено идей: 10
📌 Статус: complete
🔄 Итерация: 0
💬 Можно продолжить: True

📋 Первые 3 идеи:

--- ИДЕЯ 1 ---
ПОДАРОК: Ноутбук для работы и учебы
ПОЧЕМУ: Этот подарок идеален для подруги, так как она активна и умна. Ноутбук поможет ей продолжить 

--- ИДЕЯ 2 ---
ПОДАРОК: Квадрокоптер с камерой и пультом
ПОЧЕМУ: Относительно новый и интересный подарок, который соотв

INFO:api_qwen_kaggle:📨 Запрос: Подруге 20 лет, любит готовить и читать. Любит птиц.  Не любит сладкое и украшения., обратная связь: Убери всё, что связано с техникой. Добавь больше идей для кулинарии и чтения., итерация: 1


🔍 GIFT_AGENT START: 'Подруге 20 лет, любит готовить и читать. Любит птиц.  Не любит сладкое и украшения.' (итерация 1)
🚀 Запуск графа...
  📋 Извлечено: получатель=подруге, повод=None, интересы=['готовить', 'читать', 'любит птиц'], не любит=['сладкое', 'украшения']
  💬 Обработка обратной связи: Убери всё, что связано с техникой. Добавь больше идей для кулинарии и чтения.
    ➖ Добавлено в исключения: ['техника']
    ➕ Добавлено в интересы: ['кулинария', 'читать книги']
  💬 Обработка обратной связи: Убери всё, что связано с техникой. Добавь больше идей для кулинарии и чтения.
    ➖ Добавлено в исключения: ['техника']
    ➕ Добавлено в интересы: ['кулинария', 'чтение']


...

🔍 GIFT_AGENT START: 'вапмссвавп' (итерация 0)
🚀 Запуск графа...
  📋 Извлечено: получатель=None, повод=None, интересы=[], не любит=[]
  📋 Извлечено: получатель=папа, повод=None, интересы=['рыбалка', 'дача'], не любит=[]


INFO:pyngrok.process.ngrok:t=2026-03-20T22:05:37+0000 lvl=info msg="join connections" obj=join id=dd7e21ee5938 l=127.0.0.1:8000 r=121.127.46.69:58086


INFO:     121.127.46.69:0 - "GET /health HTTP/1.1" 200 OK


INFO:pyngrok.process.ngrok:t=2026-03-20T22:05:46+0000 lvl=info msg="join connections" obj=join id=ab60b28a95fa l=127.0.0.1:8000 r=121.127.46.69:3172
INFO:api_qwen_kaggle:📨 Запрос: 👨 Папа, 55 лет, рыбалка, дача, обратная связь: None, итерация: 0


🔍 GIFT_AGENT START: '👨 Папа, 55 лет, рыбалка, дача' (итерация 0)
🚀 Запуск графа...
  📋 Извлечено: получатель=папа, повод=None, интересы=['рыбалка', 'дача'], не любит=[]
📊 Результат: status=complete
✅ Успешно сгенерировано 8 идей
   Итерация: 0/3
